# Vehicle Price Prediction (MSRP) with Linear Regression

This notebook builds a linear regression model from scratch, using the normal equation, to predict a vehicle's manufacturer suggested retail price (MSRP) from its technical specifications.

## 1. Data Loading and Cleaning

We load the raw CSV file, standardize the column names, and normalize the text values so that categorical variables stay consistent across the dataset.

In [ ]:
import  pandas  as pd
import numpy as np
import  matplotlib.pyplot as plt
import seaborn as sns
import random

In [ ]:
data_path='data.csv'

df=pd.read_csv(data_path)

df.columns=df.columns.str.lower().str.replace(" ", "_")

# normalize text values (lowercase, spaces to underscores) so categories are not duplicated
string=list(df.dtypes[df.dtypes == 'str'].index)
for col in  string:
    df[col]=df[col].str.lower().str.replace(" ", "_")


## 2. Exploratory Data Analysis

Before modeling, we look at the shape of the data: how prices are distributed, and which vehicle makes are most represented in the dataset.

In [ ]:
for col in df.columns:
    print(col)
    print(df[col].unique()[:5])
    print(df[col].nunique())

    print("*************")

In [ ]:
sns.histplot(df.msrp[df.msrp  < 90000], bins=49)


In [ ]:
make_counts = df["make"].value_counts().sort_values()

plt.figure(figsize=(10, 12))
sns.barplot(x=make_counts.values, y=make_counts.index, color="steelblue")

plt.xlabel("Count")
plt.ylabel("Make")
plt.title("Number of Vehicles by Make")
plt.tight_layout()
plt.show()

In [ ]:
log_prices=np.log1p(df['msrp'])

sns.histplot(log_prices, bins=49)


The log-transformed price distribution is close to bell-shaped, which supports using `log1p(msrp)` as the regression target instead of the raw price.

## 3. Train / Validation / Test Split

The dataset is split into three independent parts: training data used to fit the model, validation data used to compare different feature sets, and a held out test set used only once, at the end, for the final evaluation.

In [ ]:
n=len(df)
n_valid=int(n*0.2)
n_test=int(n*0.2)
n_train=n-n_test-n_valid

print(f"({n_test},{n_train}, {n_valid})")

In [ ]:
np.random.seed(2)  # fixed seed so the split is reproducible
idx = np.arange(n)
np.random.shuffle(idx)

df_train = df.iloc[idx[:n_train]]
df_valid = df.iloc[idx[n_train:n_train+n_valid]]
df_test  = df.iloc[idx[n_train+n_valid:]]

In [ ]:
df_train.head()  # quick check that the split worked as expected

In [ ]:
df_valid=df_valid.reset_index(drop=True)
df_test=df_test.reset_index(drop=True)
df_train=df_train.reset_index(drop=True)

## 4. Preparing the Target Variable

Vehicle prices are strongly right skewed: a small number of very expensive cars sit far above the bulk of the data. Applying $\log(1+x)$ compresses this skew and brings the target closer to a normal distribution, which linear regression handles far better than a heavily skewed variable. The transform used here is $\text{log1p}(x) = \ln(1+x)$, and predictions are converted back to real prices with its inverse, $\text{expm1}(x) = e^x - 1$.

In [ ]:
y_train=np.array(np.log1p(df_train['msrp']))
y_valid=np.array(np.log1p(df_valid['msrp']))
y_test=np.array(np.log1p(df_test['msrp']))


In [ ]:
# remove the target column from the features to avoid data leakage
del df_train['msrp']
del df_valid['msrp']
del df_test["msrp"]


## 5. Linear Regression: the Normal Equation

We are looking for a weight vector $w$ that satisfies

$$X w = y$$

This system is generally overdetermined, since $X$ has more rows (observations) than columns (features), so an exact solution does not exist in general. Instead we look for the vector $w$ that minimizes the sum of squared residuals

$$J(w) = \|Xw - y\|^2 = (Xw - y)^T(Xw - y)$$

Expanding this expression gives

$$J(w) = w^T X^T X w - 2 w^T X^T y + y^T y$$

Taking the gradient with respect to $w$ and setting it to zero gives

$$\nabla_w J(w) = 2X^TXw - 2X^Ty = 0 \quad\Longrightarrow\quad X^TXw = X^Ty$$

This is the normal equation. If $X^TX$ is invertible, it has the closed form solution

$$w = (X^TX)^{-1}X^Ty$$

In the code below, a column of ones is added to $X$ so that the first entry of $w$ acts as the intercept $w_0$, and the remaining entries are the coefficients associated with each feature.

In [ ]:
def train_linear_regression(X,y):
    X=np.column_stack((np.ones(X.shape[0]),X))
    XTX=np.dot(X.T,X)
    XTX_inv=np.linalg.inv(XTX)
    w=np.dot(XTX_inv,np.dot(X.T,y))
    return w[0], w[1:]

## 6. First Model: Basic Numerical Features

We start with a small set of numeric features that are already present in the dataset, without any feature engineering, to establish a baseline.

In [ ]:
df_test.columns  # check which columns are available before picking the first feature set

In [ ]:
# first feature set: numeric columns already present in the dataset
base=['engine_hp','engine_cylinders','highway_mpg','city_mpg','popularity']
df_train=df_train.fillna(0)
X_train=df_train[base].values

In [ ]:
w0, w = train_linear_regression(X_train,y_train)
print(w0, w)

In [ ]:
predictions=w0+X_train@w
print(np.expm1(predictions[:5]))

In [ ]:
sns.histplot(predictions, bins=50, color="blue" ,alpha=0.5)
sns.histplot(y_train, bins=50, color="red", alpha=0.5)

### Limitations of the Normal Equation

This method is efficient when the number of features stays relatively small, since it requires inverting a $(p+1)\times(p+1)$ matrix, where $p$ is the number of features. This inversion becomes expensive and numerically unstable as $p$ grows, or when $X^TX$ is ill conditioned (for example when some columns are highly correlated). In those situations, iterative methods such as gradient descent are usually preferred.

In [ ]:
def rmse(y, y_pred):
    error=y-y_pred
    mse=(error**2).mean()
    return np.sqrt(mse)

In [ ]:
rmse_train=rmse(y_train,predictions)
rmse_reel=rmse(np.expm1(y_train),np.expm1(predictions))
print(rmse_train)
print(rmse_reel)

## 7. Evaluating the Model: RMSE

The RMSE, root mean squared error, measures the typical distance between predicted values $\hat{y}_i$ and true values $y_i$:

$$\text{RMSE} = \sqrt{\frac{1}{n}\sum_{i=1}^{n}(y_i - \hat{y}_i)^2}$$

where $n$ is the number of observations, $y_i$ the true value and $\hat{y}_i$ the predicted value for observation $i$.

Starting from the residual $e_i = y_i - \hat{y}_i$, squaring it removes the sign so that positive and negative errors do not cancel out, and it grows quadratically with the size of the error: an error of 10 contributes 100, while an error of 20 contributes 400, four times more, even though the error itself only doubled. This is what makes RMSE especially sensitive to large errors and outliers.

Averaging the squared errors over all observations gives the mean squared error

$$\text{MSE} = \frac{1}{n}\sum_{i=1}^{n}(y_i - \hat{y}_i)^2$$

MSE is expressed in squared units, dollars squared in this case, which has no direct real world meaning, so we take its square root to return to the original unit of $y$. This is exactly the RMSE. It is always nonnegative, equal to zero only for a perfect model, and expressed in the same unit as the target variable.

### Interpreting the Results

The training RMSE on the log scale is about 0.755, and about 92070 dollars once the predictions and targets are converted back to real prices with expm1. These two numbers describe the same error from two different angles.

The log scale RMSE captures a relative, multiplicative error, since working in log space turns ratios into differences. A value of 0.755 corresponds to a multiplicative factor of roughly $e^{0.755} \approx 2.13$, meaning predictions are typically off by about a factor of two in either direction. This is a fairly large error and shows there is room for improvement.

The real scale RMSE of 92070 dollars looks far more dramatic, but should be read carefully. Because squaring amplifies large errors, a handful of very bad predictions on expensive luxury cars or supercars is enough to push this number up sharply, even if most predictions on ordinary vehicles are reasonable. The large gap between the two numbers actually confirms that the price distribution is highly skewed, with a few extreme values dominating the error on the real scale. This is exactly why the model is trained on `log1p(msrp)` instead of the raw price, so that those few very expensive vehicles do not dominate the learning process.

To make later comparisons fair, the baseline model (numeric features only) is also evaluated on the validation set below, using the same split that will be reused for every feature addition afterward.

In [ ]:
# evaluate the baseline model (numeric features only) on the validation set as well,
# so it can be compared fairly against every model built later on, all on the same split
X_valid_base = df_valid.fillna(0)[base].values
predictions_valid_base = w0 + X_valid_base@w
rmse_valid_base = rmse(y_valid, predictions_valid_base)
print(rmse_valid_base)

## 8. Adding a Vehicle Age Feature

Instead of using the manufacturing year directly, we convert it into the age of the vehicle relative to a reference year. Age is usually a more meaningful signal for price than the raw year, since it reflects depreciation directly.

In [ ]:
base=['engine_hp','engine_cylinders','highway_mpg','city_mpg','popularity']

def prepare_X(df):
    df=df.copy()
    df['age']=2017-df.year
    df=df.fillna(0)
    features=base+['age']
    X=df[features].values
    return X

# train the model 
X_train=prepare_X(df_train)
X_valid=prepare_X(df_valid)
w0 , w = train_linear_regression(X_train,y_train)

y_pred_valid=w0+X_valid@w
rmse_valid=rmse(y_valid,y_pred_valid)
print(rmse_valid)

In [ ]:
sns.histplot(y_pred_valid, bins=50, color="blue" ,alpha=0.5)
sns.histplot(y_valid, bins=50, color="red", alpha=0.5)

### Effect of Adding Age

The baseline model, using only numeric specifications and evaluated on the validation set, scores an RMSE of about 0.762, close to its training RMSE of 0.755, which means it is not yet overfitting. Once vehicle age is added, validation RMSE drops to about 0.517, a large improvement on a fair, apples to apples comparison, since both scores come from the same validation set. This makes sense: age is closely tied to depreciation, and it carries information about price that the raw manufacturing year, once mixed with the other numeric features, did not fully capture.

## 9. Adding a Categorical Feature: Number of Doors

One hot encoding turns a categorical feature with $k$ possible values into $k$ binary columns, each indicating whether an observation belongs to that category. For a linear model, this lets every category contribute its own additive term to the prediction, instead of forcing an artificial numeric ordering between categories that are not naturally ordered.

In [ ]:
df_train.number_of_doors.unique()  # check the possible values before encoding them

In [ ]:
def prepare_X(df):
    df=df.copy()
    features=base.copy()
    df['age']=2017-df.year
    features.append('age')
    # one-hot encode the number of doors
    for v  in [2,3,4]:
        df[f'number_of_doors_{v}']=(df.number_of_doors==v).astype(int)
        features.append(f'number_of_doors_{v}')
    df=df.fillna(0)
    X=df[features].values
    return X

In [ ]:
base=['engine_hp','engine_cylinders','highway_mpg','city_mpg','popularity']
X_train=prepare_X(df_train)
w0 , w = train_linear_regression(X_train,y_train)
X_valid=prepare_X(df_valid)
predictions_valid=w0+X_valid@w
rmse_valid=rmse(y_valid,predictions_valid)
print(rmse_valid)

### Effect of Adding Number of Doors

Adding the number of doors barely moves the validation RMSE, from about 0.517 down to about 0.516. This is expected: once age and the numeric specifications are already in the model, the number of doors adds very little new information about price. Most of what it could explain is likely already captured by other features, such as vehicle size or style.

## 10. Adding a Categorical Feature: Make

The `make` column has a very large number of distinct values. Encoding every single make would add a large number of mostly sparse columns and risk overfitting on makes with very few examples, so we only encode the most frequent ones.

In [ ]:
list(df.make.value_counts().head().index)  # preview the top 5 most frequent makes before encoding them

In [ ]:
def prepare_X(df):
    df=df.copy()
    features=base.copy()
    df['age']=2017-df.year
    features.append('age')
    for v  in [2,3,4]:
        df[f'number_of_doors_{v}']=(df.number_of_doors==v).astype(int)
        features.append(f'number_of_doors_{v}')
    # keep only the 5 most frequent makes
    for make in list(df.make.value_counts().head().index):
        df[f'make_{make}']=(df.make==make).astype(int)
        features.append(f'make_{make}')
    df=df.fillna(0)
    X=df[features].values
    return X

In [ ]:
base=['engine_hp','engine_cylinders','highway_mpg','city_mpg','popularity']

X_train=prepare_X(df_train)
w0 , w = train_linear_regression(X_train,y_train)
X_valid=prepare_X(df_valid)
predictions_valid=w0+X_valid@w
rmse_valid=rmse(y_valid,predictions_valid)
print(rmse_valid)
print(rmse(np.expm1(y_valid),np.expm1(predictions_valid)))


### Effect of Adding Vehicle Make

Adding the five most frequent makes brings the validation RMSE down further, from about 0.516 to about 0.508. Even with only five brands encoded, make carries real pricing information, since it separates, at least partially, mainstream manufacturers from more premium ones. The improvement is smaller than the one brought by age, but it is consistent and in the expected direction.

## 11. Encoding All Categorical Features

Rather than adding one categorical column at a time by hand, we generalize the encoding step into a loop that handles every categorical column at once, each with its own set of frequent values.

In [ ]:
categories= ['make', 'engine_fuel_type', 'transmission_type', 'driven_wheels', 'market_category', 'vehicle_size', 'vehicle_style']

categorical_features={cat: list(df_train[cat].value_counts().head().index) for cat in categories}


In [ ]:
def prepare_X(df):
    df=df.copy()
    features=base.copy()
    df['age']=2017-df.year
    features.append('age')
    for v  in [2,3,4]:
        df[f'number_of_doors_{v}']=(df.number_of_doors==v).astype(int)
        features.append(f'number_of_doors_{v}')
    # encode every category listed in categorical_features (make is included here already)
    for cat, values in categorical_features.items():
        for v in values:
            df[f'{cat}_{v}']=(df[cat]==v).astype(int)
            features.append(f'{cat}_{v}')
    df=df.fillna(0)
    X=df[features].values
    return X



## 12. Ridge Regularization

Adding many one-hot encoded columns increases the chance that some of them are highly correlated, which makes $X^TX$ close to singular and its inversion unstable. To fix this, we add an L2 penalty term to the objective function:

$$J(w) = \|Xw - y\|^2 + r\|w\|^2$$

Taking the gradient and setting it to zero gives the ridge regression normal equation

$$(X^TX + rI)w = X^Ty$$

Adding $rI$ shifts the eigenvalues of $X^TX$ away from zero, which keeps the matrix invertible even when $X^TX$ alone is singular or nearly singular. The parameter $r$ controls the strength of this penalty: a larger $r$ shrinks the weights more aggressively toward zero, trading a bit of bias for a reduction in variance.

In [ ]:
def train_linear_regression_regulirizd(X,y,r=0.01):
    X=np.column_stack((np.ones(X.shape[0]),X))
    XTX=np.dot(X.T,X)+r*np.eye(X.shape[1])  # ridge penalty, keeps X^T X invertible even with collinear features
    XTX_inv=np.linalg.inv(XTX)
    w=np.dot(XTX_inv,np.dot(X.T,y))
    return w[0], w[1:]

In [ ]:
X_train_full=prepare_X(pd.concat([df_train,df_valid]))
y_train_full=np.concatenate([y_train,y_valid])


In [ ]:
w0 , w = train_linear_regression_regulirizd(X_train_full,y_train_full,r=0.001)
X_test=prepare_X(df_test) 
y_test_pred=w0+X_test@w
rmse_test=rmse(y_test,y_test_pred)
print(rmse_test)

## 13. Searching for the Optimal Regularization Parameter

We sweep a grid of values for $r$, retrain the model for each one, and track the validation RMSE, to find the value that gives the best trade-off between bias and variance.

In [ ]:
# sweep a grid of r values to find the best bias-variance trade-off
rmse_values=[]
xaxsix=np.linspace(0,0.2,100)
for i in xaxsix:
    w0 , w = train_linear_regression_regulirizd(X_train_full,y_train_full,r=i)
     
    y_test_pred=w0+X_test@w
    rmse_test=rmse(y_test,y_test_pred)
    rmse_values.append(rmse_test)

plt.plot(xaxsix,rmse_values)
plt.xlabel("Regularization Parameter")
plt.ylabel("RMSE")
plt.grid(True)
plt.title("RMSE vs Regularization Parameter")
plt.show()



In [ ]:
min_rmse = min(rmse_values)
optimal_r = xaxsix[rmse_values.index(min_rmse)]
optimal_r, min_rmse

### Reading the Regularization Curve

With no regularization at all ($r$ close to 0), the test RMSE spikes to around 34, an unusable result. This is the numerical instability described earlier: with every categorical column encoded at once, $X^TX$ is close to singular, and inverting it directly produces huge, unreliable weights. As soon as a small penalty is introduced, RMSE collapses back down to a reasonable range and stays essentially flat afterward. The search finds an optimal value of about $r \approx 0.002$, with a test RMSE of about 0.452, only slightly better than the very first value tried, $r = 0.001$. This confirms that the exact value of $r$ matters far less than simply having a nonzero one: the main job of regularization here is to restore numerical stability, not to fine tune performance.

## 14. Final Model on the Test Set

With the optimal regularization parameter identified, we retrain the model one last time and evaluate it on the test set, which has not been used anywhere in the process so far.

In [ ]:
# retrain using the optimal r found above, then evaluate on the test set
w0, w = train_linear_regression_regulirizd(X_train_full,y_train_full,r=optimal_r)
X_test=prepare_X(df_test)
y_test_pred=w0+X_test@w
print(rmse(y_test,y_test_pred))

## Summary

Across all the steps in this notebook, the validation and test RMSE, on the log price scale, moved as follows:

| Model | RMSE |
|---|---|
| Numeric features only (train) | 0.755 |
| + vehicle age (validation) | 0.517 |
| + number of doors (validation) | 0.516 |
| + top 5 makes (validation) | 0.508 |
| + all categorical features, ridge regularized (test) | 0.452 |

On the real price scale, the final test RMSE is about 38400 dollars. Vehicle age and vehicle make turned out to be the two most useful additions, while the number of doors contributed almost nothing. Regularization did not improve accuracy on its own, its role was to keep the normal equation numerically solvable once many correlated one-hot encoded columns were added together.

Natural next steps to push this project further would be encoding more categories instead of only the top 5, comparing this closed form solution against gradient descent or scikit-learn's `LinearRegression` and `Ridge`, and trying nonlinear models such as decision trees or gradient boosting to see how much further the RMSE can be pushed down.